## Configuration

In [ ]:
import sys
sys.path.append('../src')

## Dataset

In [ ]:
DIR_DATA = '../.data/split/'
CONDITIONS = ('neutral','stress')
DURATIONS = (0,1,2,4,6,8)
CLASS_NAMES = ('understanding', 'confusion')

from trainer import DataProvider
data = DataProvider(DIR_DATA, CONDITIONS, DURATIONS, CLASS_NAMES)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
data_validation_full = dict()
for d in DURATIONS:
    xs, ys = zip(*(data.get_validation(c, d) for c in CONDITIONS))
    x = np.concatenate(xs)
    y = np.concatenate(ys)
    data_validation_full[d] = train_test_split(x, y, test_size=0.5)

### XGBoost

In [ ]:
from models import FunctionalWrapper
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
def score(n_estimators, d):
    print(n_estimators, d)
    model = FunctionalWrapper(XGBClassifier(n_estimators=n_estimators, eval_metric='logloss', device='cuda'))
    x_train, x_eval, y_train, y_eval = data_validation_full[d]
    model.fit(x_train, y_train)
    y_pred = model.predict(x_eval)
    return f1_score(y_eval, y_pred, average='macro')

In [ ]:
ESTIMATORS = (10, 20, 50, 100, 200, 500, 1000, 2000)
scores = [
    [
        score(n, d)
        for n in ESTIMATORS
    ]
    for d in (0,1,2,4)
]

In [ ]:
import matplotlib.pyplot as plt
for d, s in zip(data.durations, scores):
    plt.plot(ESTIMATORS, s, marker='o', label=f"{d}s")
plt.ylim(0,1)
plt.xlabel('Number of Estimators')
plt.ylabel('f1-score')
plt.title('XGBoost performance')
plt.legend()
plt.show()

## Random Forests

In [ ]:
from models import FunctionalWrapper
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
def score(n_estimators, d):
    print(n_estimators, d)
    model = FunctionalWrapper(RandomForestClassifier(n_estimators, n_jobs=16))
    x_train, x_eval, y_train, y_eval = data_validation_full[d]
    model.fit(x_train, y_train)
    y_pred = model.predict(x_eval)
    return f1_score(y_eval, y_pred, average='macro')

In [ ]:
ESTIMATORS = (10, 20, 50, 100, 200, 500)
scores = [
    [
        score(n, d)
        for n in ESTIMATORS
    ]
    for d in (0,1,2,4)
]

In [ ]:
import matplotlib.pyplot as plt
for d, s in zip(data.durations, scores):
    plt.plot(ESTIMATORS, s, marker='o', label=f"{d}s")
plt.ylim(0,1)
plt.xlabel('Number of Estimators')
plt.ylabel('f1-score')
plt.title('Random Forests performance')
plt.legend()
plt.show()

No significant performance gain for more than 100 estimators

## Multi Layer Perceptron (Neural Network)

### Hyperparameter selection

f1 scoring function for parameter set
 * Builds MLP with total number of neurons devided into $n$ layers
 * Fits model on training dataset
 * Compute macro averaged f1 Score

In [ ]:
from models import MultiLabelMLP
from sklearn.metrics import f1_score
def score(n_neurons, n_layers, d=0):
    neurons_per_layer = n_neurons // n_layers
    hidden_layers = (neurons_per_layer,) * n_layers
    model = MultiLabelMLP(hidden_layers)
    x_train, x_eval, y_train, y_eval = data_validation_full[d]
    model.fit(x_train, y_train, batch_size=1000)
    y_pred = model.predict(x_eval)
    return f1_score(y_eval, y_pred, average='macro')

Search strategies Grid and Random search

In [ ]:
def grid_search(neurons, layers):
    result = dict()
    for n in neurons:
        for l in layers:
            result[(n, l)] = score(n, l)
    return result

def random_search(neurons_min, neurons_max, layers_min, layers_max, n_samples):
    result = dict()
    for i in range(n_samples):
        n = np.random.randint(neurons_min, neurons_max+1)
        l = np.random.randint(layers_min, layers_max+1)
        result[(n, l)] = score(n, l)
    return result

# Visualization
import matplotlib.pyplot as plt
def plot_search_results(results, title=''):
    p = np.array(tuple(results.keys()))
    a = np.array(tuple(results.values()))
    plt.scatter(*p.T, vmin=0, vmax=1, c=a)
    plt.xlabel('Total number of neurons')
    plt.ylabel('Number of layers')
    plt.colorbar(label='f1-score')
    plt.title(title)
    plt.show()

Perform random search

In [ ]:
rs = random_search(1, 5000, 1, 10, 50)
plot_search_results(rs, 'Random Search for MLP architecture')

Appropriate MLP acritecture requires
 * $n_{layers} \geq 3$ 
 * $n_{neurons} \geq 2000$

There is no significant change for increasing number of layers.
Thus, choose $n_{layers} = 4$

In [ ]:
neurons = (100, 200, 400, 800, 1500, 2500, 4000, 6000)
scores = np.array([
    [
        score(n, 4, d)
        for n in neurons
    ]
    for d in (0,1,2,4)
    
])
for d, s in zip(data.durations, scores):
    plt.plot(neurons, s, marker='o', label=f"{d}s")
plt.ylim(0,1)
plt.xlabel('Total number of neurons')
plt.ylabel('f1-score')
plt.title('MLP with 4 hidden layers')
plt.legend()
plt.show()


Keep number of neurons small while having good f1 score.

=> Choose $n_{neurons} = 2400$

=> MLP with 4 layers a 600 neurons